# Complexity-03g — Les trois barrières : pourquoi P ≠ NP résiste, et ce qu'une borne de circuit peut encore dire

**Série Complexity** — carnet d'accrétion `03g`, sur la base de
[03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb).
Public : **Licence**.

Ce carnet est écrit **depuis une archive datée** du site `pequalnp.com`
(« P vs NP, attacked live by a fleet of AI agents », édité par Brilliant) :
`G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - Brilliant - pequalnp.com - P vs NP attacked live by a fleet of AI agents (releve date du 25-11-2026)\`
(manifeste de provenance dans le même dossier : URL, heure UTC, sha256, taille).

Trois notions, trois régimes :

- **relativisation** (Baker–Gill–Solovay, 1975) — déjà traitée par
  [03](Complexity-03-HierarchieTemporelle-Python.ipynb) et
  [03c](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) ; **rien n'est re-démontré ici** ;
- **natural proofs** (Razborov–Rudich, 1994) — mesuré sur tables de vérité ;
- **algébrisation** (Aaronson–Wigderson, 2008) — mesuré sur degrés en interpolation.

Le quatrième objet est **le registre d'un tiers** : un protocole public où chaque
énoncé passe un vérificateur, où une réfutation est la vérification de la négation,
et où les énoncés sont parfois **retirés avec remplacement**. Ce registre est une
donnée, pas un théorème : le carnet ne le confirme pas et n'en reprend aucune
conclusion.

## 0. Pourquoi ce carnet

La série Complexity traitait la relativisation et le mur d'énumération ; elle ne
nommait ni les **natural proofs**, ni l'**algébrisation** (mesuré : 0 occurrence de
`natural proof`, `Razborov`, `Rudich`, `algebrization` dans la série avant ce carnet).
Ces deux barrières ne sont pas des variantes de la première : chacune restreint
d'une façon différente **ce qu'un argument de borne inférieure a le droit de faire**.

En 2026, une flotte d'agents a attacked P vs NP publiquement, avec un juge Lean
unique et un registre d'énoncés. Ce dispositif est un **cas d'école d'épistémologie
d'une annonce en cours** : ce qu'on peut y citer, ce qu'on ne peut pas y re-prouver,
et ce qui arrive à un énoncé quand il est réécrit.

Prérequis ajoutés par ce carnet : la notion de **table de vérité** et de
**transformée de Möbius** sur GF(2) (rappelée en §2), la **matrice d'incidence**
d'un plan projectif (rappelée en §4). Rien d'autre n'est supposé.

In [1]:
# --- Parametres (papermill) -------------------------------------------------
BATCH_MODE = "true"

import hashlib
import itertools
import math
import time
from collections import Counter
from pathlib import Path

import numpy as np

SEED = 20261125
rng = np.random.default_rng(SEED)

print("numpy", np.__version__, "| graine", SEED)

numpy 2.5.3 | graine 20261125


## 1. Les trois barrières, ce qu'elles interdisent

| Barrière | Énoncé **attribué** | Ce qu'elle interdit | Statut ici |
|---|---|---|---|
| Relativisation (Baker–Gill–Solovay, 1975) | Il existe un oracles `A` tel que `P^A = NP^A` et un oracles `B` tel que `P^B ≠ NP^B`. | Toute preuve qui ne tient pas dans **les deux** mondes. | traité en [03](Complexity-03-HierarchieTemporelle-Python.ipynb) et [03c](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) — **rappelé, pas re-démontré** |
| Natural proofs (Razborov–Rudich, STOC '94 ; JCSS 55(1), 1997) | Sous l'hypothèse de fonctions à sens unique fortes, aucun *natural proof* (grande, constructive, utile) ne prouve `P ≠ NP`. | Une borne inférieure « naturelle » (densité ≥ 1/2, calculable en temps polynomial depuis la table de vérité) **utile** pour une borne de circuits. | **mesuré** en §2 : deux premières colonnes, la troisième est le mur |
| Algébrisation (Aaronson–Wigderson, STOC 2008 ; version étendue TOCT 1(1), 2009) | Si `P ≠ NP` alors `P ≠ NP` **relativisé à un oracle algébrique** (l'oracle calcule des polynômes multilinéaires sur GF(p)). | Une preuve qui survit à la relativisation mais **pas** à l'algébrisation. | **mesuré** en §3 : degrés d'interpolation, puis un jouet d'oracles |

**Trois régimes, pas trois variantes.** La relativisation dit « la preuve doit
tenir dans deux mondes » ; les natural proofs disent « la preuve doit être
naturelle *et* ne peut pas l'être » ; l'algébrisation ajoute un troisième monde
où l'oracle est **calculable par un polynôme**. Un énoncé de la série
[03d](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) — l'ETH et sa
réfutation — se lit differently dans chacun des trois régimes.

In [2]:
# Les trois barrieres : un en-tete execute, pour que la table ci-dessus ne soit
# qu'une projection d'une structure unique (nom, annee, source, regime).
BARRIERES = [
    {
        "nom": "relativisation",
        "auteurs": "Baker, Gill, Solovay",
        "annee": 1975,
        "source": "SIAM J. Comput. 4(4)",
        "regime": "deux mondes",
        "traitee_ailleurs": "03 et 03c de cette serie",
    },
    {
        "nom": "natural proofs",
        "auteurs": "Razborov, Rudich",
        "annee": 1994,
        "source": "STOC '94 ; J. Comput. Syst. Sci. 55(1) 1997",
        "regime": "le monde naturel est mur",
        "traitee_ailleurs": "mesure ici (section 2)",
    },
    {
        "nom": "algebrization",
        "auteurs": "Aaronson, Wigderson",
        "annee": 2008,
        "source": "STOC 2008 ; version etendue TOCT 1(1) 2009",
        "regime": "un troisieme monde polynomial",
        "traitee_ailleurs": "mesure ici (section 3)",
    },
]

for b in BARRIERES:
    print(f"{b['nom']:>18} ({b['annee']})  regime : {b['regime']:<28}  -> {b['traitee_ailleurs']}")

print()
print("Registre natif des trois sources :", "; ".join(f"{b['source']}" for b in BARRIERES))

    relativisation (1975)  regime : deux mondes                   -> 03 et 03c de cette serie
    natural proofs (1994)  regime : le monde naturel est mur      -> mesure ici (section 2)
     algebrization (2008)  regime : un troisieme monde polynomial  -> mesure ici (section 3)

Registre natif des trois sources : SIAM J. Comput. 4(4); STOC '94 ; J. Comput. Syst. Sci. 55(1) 1997; STOC 2008 ; version etendue TOCT 1(1) 2009


## 2. Natural proofs, mesurés sur des tables de vérité

Une **propriété** d'une famille de fonctions booléennes est dite :

- **grande** (*large*) si elle vaut `vrai` pour au moins une fraction `1/2` des
  fonctions ;
- **constructive** si, donnée la table de vérité, on peut décider la propriété en
  temps polynomial en `n` ;
- **utile** (*useful*) si elle ne se réduit pas à une tautologie et peut servir à
  établir une borne inférieure de circuits.

Les trois conditions sont **attribuées** à Razborov–Rudich (1994) ; le théorème
est **cité, non re-démontré** ici. Ce que le carnet mesure, ce sont les deux
premières colonnes, et il **nomme** la troisième comme le mur.

### Témoin 1 — la densité et le coût, colonne par colonne

Sur **toutes** les fonctions booléennes à `n = 4` variables — la population
complète, `2^16 = 65 536` fonctions de 16 entrées — on mesure : le **degré
algébrique** (transformée de Möbius sur GF(2)), la **sensibilité moyenne** (on
retourne chaque entrée), et la parité. Les densités sont donc **exactes**, pas
estimées.

In [3]:
def _moebius_sur_lignes(a: np.ndarray) -> np.ndarray:
    """Transformee de Moebius sur GF(2), appliquee a chaque ligne.

    a : (N, 2**n) tableau de bits. Retourne les coefficients de l'ANF
    (a[S] = XOR des f(T) pour T sous-ensemble de S).
    """
# Transformation de Moebius, en place : a chaque passe, on echange
# les deux moities d'une variable -- le « papillon » de l'ANF.
    a = a.astype(np.uint8).copy()
    n = a.shape[1].bit_length() - 1
    for i in range(n):
        step = 1 << i
        vue = a.reshape(a.shape[0], -1, 2, step)
# Au pas i, on additionne la moitie basse dans la moitie haute de chaque
# bloc de 2**i entrees (en GF(2), l'addition est un XOR) : c'est
# l'inversion de Moebius, c'est-a-dire c[S] = XOR des f(T) pour T inclus
# dans S. Le sens inverse calculerait des sommes sur les sur-ensembles,
# qui ne sont pas l'ANF.
        vue[:, :, 1, :] ^= vue[:, :, 0, :]
        a = vue.reshape(a.shape[0], -1)
    return a


def degres_algbriques(fns: np.ndarray) -> np.ndarray:
    """Degres algebriques (ANF sur GF(2)) de chaque table de verite."""
# L'ANF s'obtient par la meme transformee, ligne par ligne.
    anf = _moebius_sur_lignes(fns)
    n = fns.shape[1].bit_length() - 1
# Poids d'un monome = nombre de variables qu'il contient : le degre
# algebrique est le plus grand poids present dans l'ANF.
    poids = np.array([bin(s).count("1") for s in range(2 ** n)], dtype=np.int32)
    degres = np.zeros(fns.shape[0], dtype=np.int32)
    for i in range(fns.shape[0]):
        support = np.flatnonzero(anf[i] == 1)
        degres[i] = poids[support].max() if support.size else 0
    return degres


def sensibilite_moyenne(fns: np.ndarray) -> np.ndarray:
    """Sensibilite moyenne : fraction des entrees dont le retournement change la sortie.

    On compare f(x) et f(x xor 2^i) -- la VRAIE voisine -- et non pas la sortie
    retournee : c'est l'erreur qui ferait croire toute fonction sensible.
    Chaque arete du cube est rencontree deux fois (une par chaque bout), d'ou
    la division par 2 ; le cube a 4 bits a 32 aretes.
    """
    n = fns.shape[1].bit_length() - 1
# Chaque arete du cube est rencontree deux fois (une par chaque bout) :
# d'ou la division par 2, qui compte les 2*n entrees, non 2*n aretes.
    idx = np.arange(fns.shape[1])
    cumule = np.zeros(fns.shape[0], dtype=np.int32)
    for i in range(n):
        voisine = fns[:, idx ^ (1 << i)]
        cumule += (fns != voisine).sum(axis=1)
    return cumule / (2 * n)


def toutes_fonctions(n: int) -> np.ndarray:
    """Population COMPLETE des fonctions booleennes a n variables.

    table[g][x] vaut le bit x de g : c'est la fonction g evaluee sur
    l'affectation x. Pour n = 4, les 2^16 = 65 536 fonctions, en tables de
    16 bits -- la population entiere, donc des densites exactes.
    """
    idx_f = np.arange(2 ** (2 ** n))
    idx_x = np.arange(2 ** n)
# La population COMPLETE des fonctions booleennes a n variables :
# 2**(2**n) tables de 2**n entrees. Les densites qui suivent sont
# donc exactes, pas estimees.
    return ((idx_f[:, None] >> idx_x[None, :]) & 1).astype(np.uint8)


n = 4
tables = toutes_fonctions(n)
print(f"n = {n} variables -> population complete : {tables.shape[0]} fonctions de {tables.shape[1]} entrees")

# Mesures sur la population complete : degre algebrique et sensibilite.
degres = degres_algbriques(tables)
sensib = sensibilite_moyenne(tables)

print("degre maximal :", int(degres.max()))
print("fonctions de degre n (la parite et ses proches) :", int(np.sum(degres == n)))
print("densite parite (deg == n) :", float(np.mean(degres == n)))

n = 4 variables -> population complete : 65536 fonctions de 16 entrees


degre maximal : 4
fonctions de degre n (la parite et ses proches) : 32768
densite parite (deg == n) : 0.5


### Lecture du résultat — deux colonnes mesurées, la troisième nommée

La parité a un **degré algébrique maximal** et une **densité exactement 1/2** :
elle est grande au sens de Razborov–Rudich. Elle est aussi **constructive** : le
degré se calcule depuis la seule table de vérité, en `O(n · 2^n)` opérations,
sans rien supposer de la structure du circuit.

Mais la parité est **inutile** pour une borne de circuits : elle est vraie
exactement pour une fonction sur deux, et son complémentaire est faux pour une
fonction sur deux. C'est là que se place le mur du théorème : une propriété
naturelle doit être **grande** *et* **constructive** *et* **utile**, et c'est la
troisième condition que le théorème rend inatteignable sous l'hypothèse de
fonctions à sens unique fortes.

Le tableau ci-dessous mesure les deux premières colonnes sur les mêmes 256
fonctions ; la troisième est **hors de portée**, et le carnet le dit.

In [4]:
# --- Témoin 1 : le tableau des propriétés, densite + cout de calcul ---------
def pseudo_famille(n: int, k: int) -> np.ndarray:
    """Famille pseudo-aleatoire : premier bit de SHA-256(cle || x).

    Sert de témoin negatif : les tests « naturels » ne doivent pas la
    distinguer d'une famille tiree au hasard.
    """
    bits = np.zeros((k, 2 ** n), dtype=np.uint8)
    for j in range(k):
        cle = hashlib.sha256(f"{SEED}-{j}".encode("utf-8")).digest()
        for x in range(2 ** n):
            d = hashlib.sha256(cle + bytes([x & 0xFF, (x >> 8) & 0xFF])).digest()
            bits[j, x] = d[0] & 1
    return bits


def densites(fns: np.ndarray, degres_fns: np.ndarray, sensib_fns: np.ndarray) -> dict:
    """Densites des propriétés mesurees, sur un lot de tables de verite."""
    n = fns.shape[1].bit_length() - 1
    out = {}
    out["parite (deg == n)"] = float(np.mean(degres_fns == n))
    for d in range(1, n + 1):
        out[f"deg >= {d}"] = float(np.mean(degres_fns >= d))
    for s in (2, 4, 6):
        out[f"sensibilite moyenne >= {s}"] = float(np.mean(sensib_fns >= s))
    return out


mesurees = densites(tables, degres, sensib)

# Cout de calcul : le prix pour DECIDER la propriété depuis la table de verite.
t0 = time.perf_counter()
for _ in range(20):
    _ = degres_algbriques(tables)
t_degree = (time.perf_counter() - t0) / 20
t0 = time.perf_counter()
for _ in range(20):
    _ = sensibilite_moyenne(tables)
t_sensib = (time.perf_counter() - t0) / 20

print(f"{'propriete':<34} {'densite':>9}  {'constructive ?':<14} utile ?")
print("-" * 78)
for cle, val in mesurees.items():
    print(f"{cle:<34} {val:>9.4f}  {'oui':<14} {'hors de portee' if cle == 'parite (deg == n)' else '?'}")

print()
print(f"cout decideur de degre  : {t_degree * 1e3:8.2f} ms pour {tables.shape[0]} tables ({n} variables)")
print(f"cout decideur de sensib.: {t_sensib * 1e3:8.2f} ms pour {tables.shape[0]} tables ({n} variables)")

# --- Temoin negatif : la famille pseudo-aleatoire ---------------------------
K = 256
famille = pseudo_famille(n, K)
degr_pseudo = degres_algbriques(famille)
sens_pseudo = sensibilite_moyenne(famille)
mes_pseudo = densites(famille, degr_pseudo, sens_pseudo)

print()
print(f"Comparaison population complete ({tables.shape[0]} fonctions) "
      f"/ famille pseudo-aleatoire (K = {K}) :")
PROPRIETES = ("parite (deg == n)", "deg >= 3", "sensibilite moyenne >= 6")
sigma_obs = {}
for cle in PROPRIETES:
    p_pop = mesurees[cle]
    p_pseudo = mes_pseudo[cle]
    et = math.sqrt(p_pop * (1 - p_pop) / K)
    sigma = (p_pseudo - p_pop) / et if et > 0 else float("nan")
    sigma_obs[cle] = sigma
    print(f"  {cle:<30} population = {p_pop:.4f} | pseudo = {p_pseudo:.4f} | ecart = {sigma:+.2f} sigma")

# --- Reference : la loi des memes ecarts pour un tirage ALEATOIRE ---------
# Constater trois ecarts ne dit rien tant qu'on ne sait pas a quoi les
# comparer. On tire R fois K fonctions au hasard dans la population
# complete, et on mesure la loi des memes ecarts. Semence fixee.

class SplitMix64:
    """PRNG SplitMix64 : les memes tirages en Python et en C#.

    Six lignes, 64 bits, et le meme etat final des deux cotes -- c'est
    ce qui rend la parite des jumeaux totale, y compris sur la loi de
    reference.
    """

    def __init__(self, graine: int) -> None:
        self.etat = graine & 0xFFFFFFFFFFFFFFFF

    def suivant(self) -> int:
        self.etat = (self.etat + 0x9E3779B97F4A7C15) & 0xFFFFFFFFFFFFFFFF
        z = self.etat
        z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & 0xFFFFFFFFFFFFFFFF
        z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & 0xFFFFFFFFFFFFFFFF
        return z ^ (z >> 31)


R = 200
prng = SplitMix64(SEED)
ecarts_ref = {cle: [] for cle in PROPRIETES}
for _ in range(R):
    # 65 536 = 2**16 divise 2**64 : le modulo est donc sans biais.
    idx = np.array([prng.suivant() % tables.shape[0] for _ in range(K)], dtype=np.int64)
    tirage = tables[idx]
    mes_tirage = densites(tirage, degres_algbriques(tirage), sensibilite_moyenne(tirage))
    for cle in PROPRIETES:
        p_pop = mesurees[cle]
        et = math.sqrt(p_pop * (1 - p_pop) / K)
        ecarts_ref[cle].append((mes_tirage[cle] - p_pop) / et if et > 0 else float("nan"))

print()
print(f"Reference : {R} tirages aleatoires de K = {K} fonctions dans la population complete")
print(f"{'propriete':<30} {'ecart-type':>12} {'part |ecart| >= observe':>24} {'observe':>12}")
print("-" * 82)
for cle in PROPRIETES:
    vals = np.array(ecarts_ref[cle])
    part = float(np.mean(np.abs(vals) >= abs(sigma_obs[cle])))
    print(f"{cle:<30} {vals.std(ddof=1):>12.2f} {part:>23.1%} {sigma_obs[cle]:>+11.2f} sigma")


propriete                            densite  constructive ? utile ?
------------------------------------------------------------------------------
parite (deg == n)                     0.5000  oui            hors de portee
deg >= 1                              1.0000  oui            ?
deg >= 2                              0.9995  oui            ?
deg >= 3                              0.9688  oui            ?
deg >= 4                              0.5000  oui            ?
sensibilite moyenne >= 2              0.9985  oui            ?
sensibilite moyenne >= 4              0.6566  oui            ?
sensibilite moyenne >= 6              0.0080  oui            ?

cout decideur de degre  :   407.61 ms pour 65536 tables (4 variables)
cout decideur de sensib.:    13.59 ms pour 65536 tables (4 variables)

Comparaison population complete (65536 fonctions) / famille pseudo-aleatoire (K = 256) :
  parite (deg == n)              population = 0.5000 | pseudo = 0.5195 | ecart = +0.62 sigma
  deg >= 3 


Reference : 200 tirages aleatoires de K = 256 fonctions dans la population complete
propriete                        ecart-type  part |ecart| >= observe      observe
----------------------------------------------------------------------------------
parite (deg == n)                      1.03                   58.0%       +0.62 sigma
deg >= 3                               1.01                   38.0%       +1.08 sigma
sensibilite moyenne >= 6               1.02                   48.0%       -0.73 sigma


### Lecture du résultat — trois écarts de l'ordre de l'unité, comme un tirage aléatoire

Les mêmes propriétés, mesurées sur une famille tirée de `SHA-256` (K = 256
échantillons), donnent des densités **comparables** à celles de la population :
les écarts valent **+0,62 σ**, **+1,08 σ** et **−0,73 σ**, en unités de
l'écart-type d'un tirage de 256 fonctions.

Constater trois écarts ne suffirait pas à dire que la famille ne se distingue
pas : il faudrait la loi contre laquelle ils se lisent. Le carnet la **mesure** —
il tire **200 fois** 256 fonctions au hasard dans la population complète et
regarde la loi des mêmes écarts. Cette loi a un écart-type de **1,03**, **1,01**
et **1,02**, et les trois écarts observés sont atteints ou dépassés par
**58,0 %**, **38,0 %** et **48,0 %** des tirages aléatoires : ils sont
**ordinaires**. Aucun des trois ne sort de ce qu'un tirage au hasard produit
couramment.

Les tests « naturels » acceptent donc cette famille comme ils accepteraient du
hasard. C'est exactement le point du troisième régime, et c'est pourquoi la
troisième colonne du tableau reste **hors de portée** : trouver une propriété
grande, constructive **et** utile serait précisément construire ce que le
théorème attribué rend impossible sous hypothèse forte.

**Ce que ce carnet ne fait pas** : il ne prouve pas le théorème de
Razborov–Rudich, il ne l'attribue pas non plus comme une conséquence de la
mesure ci-dessus. La mesure porte sur les **deux premières colonnes** ;
l'hypothèse (existence de fonctions à sens unique fortes) est **attribuée** et
non vérifiée ici.

## 3. Algébrisation : le degré d'un polynôme multilinéaire

L'extension multilinéaire d'une fonction `f : {0,1}^n → F_p` sur un corps `F_p`
est l'unique polynôme multilinéaire `f̃` tel que `f̃(1_T) = f(T)` pour tout
sous-ensemble `T ⊆ [n]`. Dans la base des « boîtes », ses coefficients s'obtiennent
par inversion de Möbius :

```
c_S = Σ_{T ⊆ S} (-1)^{|S|-|T|} · f(1_T)     (mod p)
```

Sur GF(2), cette formule redonne l'ANF de la section 2 ; sur un corps quelconque,
les coefficients sont des différences. Le **degré** de `f̃` est le plus grand `|S|`
tel que `c_S ≠ 0`.

Attention : ce degré **dépend du corps**. Sur GF(2) — l'ANF usuelle — le degré est
un invariant de la fonction booléenne ; sur GF(p) avec `p` impair, l'extension
multilinéaire d'une fonction **booléenne** (à valeurs dans `{0,1}`) peut avoir un
tout autre degré. C'est mesuré ci-dessous, et c'est un fait important : le « degré
algébrique » n'est pas un invariant de la fonction, mais un invariant de la
fonction **vue dans un corps**.

Le tableau mesure donc deux choses : le degré (sur GF(2), GF(3), GF(5)) et le
**nombre de portes** d'un circuit explicite en portes AND/OR/NOT — un circuit
**construit** puis **compté**, jamais une porte affirmée.

In [5]:
def _moebius_mod_p(a: np.ndarray, p: int) -> np.ndarray:
    """Inversion de Moebius sur Z/p, ligne par ligne.

    a : (N, 2**n) coefficients (0..p-1). Retourne les coefficients de
    l'extension multilineaire.
    """
    a = a.astype(np.int64).copy() % p
    n = a.shape[1].bit_length() - 1
    for i in range(n):
        step = 1 << i
        vue = a.reshape(a.shape[0], -1, 2, step)
        vue[:, :, 1, :] = (vue[:, :, 1, :] - vue[:, :, 0, :]) % p
        a = vue.reshape(a.shape[0], -1)
    return a


def degree_extension(f: np.ndarray, p: int) -> int:
    """Degré de l'extension multilineaire sur GF(p) d'une table de verite."""
    c = _moebius_mod_p(f.reshape(1, -1), p)[0]
    n = f.shape[0].bit_length() - 1
    poids = np.array([bin(s).count("1") for s in range(2 ** n)], dtype=np.int32)
    support = np.flatnonzero(c != 0)
    return int(poids[support].max()) if support.size else 0


def fonctions_de_reference(n: int) -> dict:
    """AND, OR, PARITY, MAJORITY sur n variables, en tables de verite."""
    idx = np.arange(2 ** n)
    bits = np.array([(idx >> i) & 1 for i in range(n)], dtype=np.uint8)
    majorite = (bits.sum(axis=0) > n // 2).astype(np.uint8)
    return {
        "AND": np.prod(bits, axis=0).astype(np.uint8),
        "OR": (bits.sum(axis=0) > 0).astype(np.uint8),
        "PARITY": (bits.sum(axis=0) % 2).astype(np.uint8),
        "MAJORITY": majorite,
    }


class Circuit:
    """Compteur de portes AND/OR/NOT : le circuit est CONSTRUIT, puis compte.

    Une reference est soit un nom d'entree (chaine), soit ("g", index).
    """

    def __init__(self, entrees):
        self.entrees = list(entrees)
        self.gates = []

    def AND(self, a, b):
        self.gates.append(("AND", a, b))
        return ("g", len(self.gates) - 1)

    def OR(self, a, b):
        self.gates.append(("OR", a, b))
        return ("g", len(self.gates) - 1)

    def NOT(self, a):
        self.gates.append(("NOT", a, None))
        return ("g", len(self.gates) - 1)

    def XOR(self, a, b):
        """XOR en 5 portes : NOT a, NOT b, AND, AND, OR."""
        na, nb = self.NOT(a), self.NOT(b)
        return self.OR(self.AND(a, nb), self.AND(na, b))

    def MAJ3(self, a, b, c):
        """Majorite de trois : (a AND b) OR (c AND (a OR b)) -- 4 portes."""
        return self.OR(self.AND(a, b), self.AND(c, self.OR(a, b)))

    @property
    def n_portes(self) -> int:
        return len(self.gates)


n = 7  # impair : le degre de MAJORITY n'est PAS n -- il est mesure dans le tableau
def _evaluer(circuit: Circuit, sortie, affectation: dict) -> int:
    """Evalue un circuit sur une affectation des entrees (0 ou 1)."""
    env: dict = dict(affectation)
    for i, (kind, a, b) in enumerate(circuit.gates):
        va = env[a] if isinstance(a, str) else env[a[1]]
        if kind == "AND":
            vb = env[b] if isinstance(b, str) else env[b[1]]
            env[i] = va & vb
        elif kind == "OR":
            vb = env[b] if isinstance(b, str) else env[b[1]]
            env[i] = va | vb
        else:
            env[i] = 1 - va
    return env[sortie[1]] if isinstance(sortie, tuple) else env[sortie]


def _circuit_seuil(entrees: list[str], k: int) -> tuple[Circuit, tuple]:
    """Circuit du seuil « au moins k entrees a 1 ».

    Construction directe : la disjonction des C(n, k) sous-ensembles de
    taille k -- un sous-ensemble compte quand toutes ses entrees sont a 1,
    et il en existe un exactement quand au moins k entrees sont a 1. C'est
    une borne haute de construction, pas un optimum : le carnet compte, il
    n'affirme pas.
    """
    from itertools import combinations

    c = Circuit(entrees)
    termes = []
    for S in combinations(range(len(entrees)), k):
        and_s = entrees[S[0]]
        for i in S[1:]:
            and_s = c.AND(and_s, entrees[i])
        termes.append(and_s)
    sortie = termes[0]
    for terme in termes[1:]:
        sortie = c.OR(sortie, terme)
    return c, sortie


refs = fonctions_de_reference(n)

# Circuits explicites : le nombre de portes est COMPTE, jamais affirme.
c = Circuit([f"x{i}" for i in range(n)])
t = "x0"
for i in range(1, n):
    t = c.AND(t, f"x{i}")
n_and = c.n_portes

c = Circuit([f"x{i}" for i in range(n)])
t = "x0"
for i in range(1, n):
    t = c.OR(t, f"x{i}")
n_or = c.n_portes

c = Circuit([f"x{i}" for i in range(n)])
t = "x0"
for i in range(1, n):
    t = c.XOR(t, f"x{i}")
n_parity = c.n_portes

c_maj, sortie_maj = _circuit_seuil([f"x{i}" for i in range(n)], (n + 1) // 2)
n_maj = c_maj.n_portes

c = Circuit(["x0", "x1"])
c.AND("x0", "x1")
n_and2 = c.n_portes

print(f"n = {n} variables (impair) -- degre algebrique contre nombre de portes")
print(f"{'fonction':<22}" + "".join(f"{'deg p=' + str(p):>11}" for p in (2, 3, 5)) + f"{'portes':>9}")
print("-" * 66)
for nom, f, nb in (
    ("AND (n variables)", refs["AND"], n_and),
    ("OR (n variables)", refs["OR"], n_or),
    ("PARITY (n variables)", refs["PARITY"], n_parity),
    ("MAJORITY (n variables)", refs["MAJORITY"], n_maj),
):
    print(f"{nom:<22}" + "".join(f"{degree_extension(f, p):>11}" for p in (2, 3, 5)) + f"{nb:>9}")

f_and2 = np.array([0, 0, 0, 1], dtype=np.uint8)  # x0 AND x1 sur 2 variables
print(f"{'AND_2 (une seule porte)':<22}"
      + "".join(f"{degree_extension(f_and2, p):>11}" for p in (2, 3, 5)) + f"{n_and2:>9}")

# Controle : le circuit MAJORITY calcule-t-il vraiment la fonction ? On le
# verifie sur les 2**n affectations -- on ne l'affirme pas.
n_ok = 0
for masque in range(1 << n):
    affectation = {f"x{i}": (masque >> i) & 1 for i in range(n)}
    attendu = 1 if sum(affectation.values()) >= (n + 1) // 2 else 0
    n_ok += _evaluer(c_maj, sortie_maj, affectation) == attendu
print(f"controle : le circuit MAJORITY calcule la fonction sur {n_ok} / {1 << n} affectations")

# Verification de la construction sur un cas dont la reponse est connue :
# l'indicateur de la fonction z1 z2 z3 a pour ANF z1 z2 z3 (degre 3).
f = np.zeros(8, dtype=np.uint8)
for x in range(8):
    z = [(x >> i) & 1 for i in range(3)]
    f[x] = z[0] * z[1] * z[2]
print()
print("controle : degre de z1 z2 z3 =", degree_extension(f, 2), "(attendu 3)")

n = 7 variables (impair) -- degre algebrique contre nombre de portes
fonction                  deg p=2    deg p=3    deg p=5   portes
------------------------------------------------------------------
AND (n variables)               7          7          7        6
OR (n variables)                7          7          7        6
PARITY (n variables)            1          7          7       30
MAJORITY (n variables)          4          7          5      139
AND_2 (une seule porte)          2          2          2        1
controle : le circuit MAJORITY calcule la fonction sur 128 / 128 affectations

controle : degre de z1 z2 z3 = 3 (attendu 3)


### Lecture du résultat — deux degrés pour la même fonction booléenne

Trois faits se lisent dans le tableau :

- **sur GF(2)** (l'ANF), `AND_7` et `OR_7` sont de degré 7, `PARITY_7` de degré **1**
  — une fonction *linéaire* ! — et `MAJORITY_7` de degré 4 ;
- **sur GF(3)**, les quatre remontent à 7 : la **même fonction booléenne** a un
  **degré différent** selon le corps ;
- **le nombre de portes ne suit pas le degré** : `AND_2` est **une seule porte**
  et porte un degré 2, quand `PARITY_7` coûte 30 portes pour un degré 1 sur GF(2).
  Le seuil `MAJORITY_7` coûte **139 portes** dans la construction directe du
  carnet — la disjonction des 35 quadruplets — dont le calcul de la fonction
  est **contrôlé sur les 128 affectations** : un seuil coûte bien plus cher
  qu'une conjonction, et ce nombre est un majorant de la construction, pas un
  optimum.

Le troisième point est le plus utile : le degré n'est **pas** une borne inférieure
sur la taille des circuits — `AND_2` le disprose en une ligne. Un invariant calculé
dans un corps ne se transporte donc pas d'un monde à l'autre : c'est exactement la
raison d'être d'un monde où l'oracle calcule des polynômes sur un corps **choisi**.

Le **témoin 2** est cette table. Reste le **jouet d'oracles** : pourquoi la
relativisation ne se transporte-t-elle pas telle quelle au monde algébrique ? On le
mesure sur un cas concret.

In [6]:
# --- Temoin 2 bis : le jouet d'oracles --------------------------------------
# Deux mondes de MEME cardinalite, dans le MEME espace : GF(2)^4 prive de {0}, 15 points.
# Monde A : le plan de Fano, transporte (7 points, 7 droites).
# Monde B : 7 points tires au hasard dans les 15.
# Un test NATUREL (compter les points) ne les distingue jamais ; un test
# ALGEBRIQUE (x + y + z = 0) en donne des comptes tres differents.

def triplets_nuls(S) -> int:
    """Nombre de triplets {x, y, z} distincts de S tels que x + y + z = 0.

    Chaque triplet est compte par ses trois paires, d'ou la division.
    """
    S = set(S)
    paires = sum(1 for a in S for b in S if a < b and (a ^ b) in S)
    return paires // 3


fano = sorted(v << 1 for v in range(1, 8))       # plan de Fano dans GF(2)^4
ambiant = list(range(1, 16))                      # GF(2)^4 prive de {0}
comptages = []
for _ in range(2000):
    tirage = sorted(rng.choice(ambiant, size=7, replace=False).tolist())
    comptages.append(triplets_nuls(tirage))
comptages = np.array(comptages)

print(f"monde A -- plan de Fano ({len(fano)} points) : {triplets_nuls(fano)} triplets x + y + z = 0")
print(f"monde B -- 7 points tires au hasard          : moyenne {comptages.mean():.3f}, "
      f"max {comptages.max()}, part des tirages a {triplets_nuls(fano)} : "
      f"{np.mean(comptages == triplets_nuls(fano)):.4f}")

# Test naturel : le nombre de points. Il ne distingue RIEN.
print()
print(f"test naturel (cardinalite)  : monde A = {len(fano)} points, monde B = 7 points -> indistinguables")

# Degre algebrique de l'indicateur dans les deux mondes.
def indicateur(ensemble, n_bits: int) -> np.ndarray:
    f = np.zeros(2 ** n_bits, dtype=np.uint8)
    for x in range(2 ** n_bits):
        f[x] = 1 if x in set(ensemble) else 0
    return f


deg_a = degree_extension(indicateur(fano, 4), 2)
deg_b = degree_extension(indicateur(rng.choice(ambiant, size=7, replace=False).tolist(), 4), 2)
print(f"degre algebrique de l'indicateur : monde A = {deg_a} | monde B = {deg_b}")
print("=> le DEGRE ne distingue pas les deux mondes ; la STRUCTURE algebrique, oui.")

monde A -- plan de Fano (7 points) : 7 triplets x + y + z = 0
monde B -- 7 points tires au hasard          : moyenne 2.675, max 7, part des tirages a 7 : 0.0035

test naturel (cardinalite)  : monde A = 7 points, monde B = 7 points -> indistinguables
degre algebrique de l'indicateur : monde A = 4 | monde B = 4
=> le DEGRE ne distingue pas les deux mondes ; la STRUCTURE algebrique, oui.


### Lecture du résultat — deux mondes de même densité, un seul se distingue

Les deux mondes ont la **même densité** (7 points sur 15) et le **même degré**
algebrique de leur indicateur : aucun test « naturel » ne les sépare, exactement
comme dans la section 2. Un test qui pose la question *algébrique* — le nombre de
triplets `x + y + z = 0` — les sépare presque toujours : le plan de Fano en a 7,
un tirage aléatoire en a **2,7 en moyenne**, et n'atteint 7 que dans 0,35 % des
tirages (mesuré sur 2 000 tirages).

C'est le sens de l'algébrisation : **un monde supplémentaire** où l'oracle est
polynomial. Une preuve doit y survivre aussi. Le théorème d'Aaronson–Wigderson est
**cité, non démontré** ; ce jouet en montre la raison d'être, et il ne prétend
rien sur `P ≠ NP`.

## 4. Bornes inférieures de circuits : du comptage à l'explicite

Trois paliers, du plus faible au plus concret :

1. **le comptage** (Shannon) — combien de fonctions sont *au plus* atteignables
   avec `s` portes ? On compare le **décompte théorique** des configurations de
   circuits à la **clôture exhaustive** des fonctions réellement atteignables ;
2. **l'objet explicite** — le plan de Fano et son graphe d'incidence de Heawood :
   14 sommets, 21 arêtes, 3-régulier, sans 4-cycle ;
3. **la borne du registre** — une forme restrictive (forme **étoile**) dont le
   protocole public énonce une formule en `|E|` et `τ` : `|E| + τ − 1` portes.
   Cette formule est **attribuée** au registre du site archivé, avec sa
   restriction et sa date de relevé ; elle est **ici recalculée** sur des
   graphes où le calcul est entier et vérifiable.

Le graphe est ici un **graphe simple** : `E` est l'ensemble des arêtes, `τ` le
taille d'un couplage maximum (matching maximum).

In [7]:
def _portee(n: int, s: int) -> int:
    """Decompte (majorant) des configurations de circuits de s portes.

    Portes AND/OR (fan-in 2) et NOT (fan-in 1), signaux disponibles : les n
    entrees et les s sorties de portes. Chaque porte : 3 types et au plus
    (n+s)^2 choix de fils. Majorant volontairement grossier.
    """
    return 3 ** s * (n + s) ** (2 * s)


def cloture(n: int, k: int) -> dict:
    """Fonctions booleennes a n variables atteignables par <= k portes.

    BFS sur les ensembles de fonctions (representes par un entier de n bits).
    Retourne le nombre de fonctions par niveau et le nombre maximal de portes
    necessaires pour une fonction quelconque.
    """
    mask = (1 << (1 << n)) - 1
    niveau = set()
    for i in range(n):
        niveau.add(sum(((x >> i) & 1) << x for x in range(1 << n)))
    vus = set(niveau)
    par_niveau = {0: len(vus)}
    courant = niveau
    for p in range(1, k + 1):
        nouveau = set()
        for a in courant:
            nouveau.add((~a) & mask)
            for b in courant:
                nouveau.add(a & b)
                nouveau.add(a | b)
        nouveau -= vus
        vus |= nouveau
        par_niveau[p] = len(nouveau)
        courant = nouveau
        if not courant:
            break
    return {"par_niveau": par_niveau, "total": len(vus)}


n = 3
print(f"n = {n} variables -> {2 ** (2 ** n)} fonctions booleennes au total")
cl = cloture(n, 4)
cumul = 0
for p, nb in cl["par_niveau"].items():
    cumul += nb
    print(f"  <= {p} porte(s) : +{nb:>4} fonctions atteintes (cumul {cumul})")

print()
print("Comparaison avec le decompte theorique des configurations :")
for s in range(1, 5):
    print(f"  s = {s} : configurations majorantes = {_portee(n, s):>10} "
          f"(fonctions totales = {2 ** (2 ** n)})")

# La borne de comptage : il faut que le decompte des configurations depasse
# le nombre de fonctions pour qu'aucune fonction ne soit forcee d'exiger s portes.
s = 1
while _portee(n, s) < 2 ** (2 ** n):
    s += 1
print()
print(f"borne de comptage (majorant grossier) : une fonction exige au moins {s} portes")
print(f"cloture exhaustive mesuree : {cl['total']} fonctions sur {2 ** (2 ** n)} sont atteignables en <= 4 portes")

n = 3 variables -> 256 fonctions booleennes au total
  <= 0 porte(s) : +   3 fonctions atteintes (cumul 3)
  <= 1 porte(s) : +   9 fonctions atteintes (cumul 12)
  <= 2 porte(s) : +  40 fonctions atteintes (cumul 52)
  <= 3 porte(s) : + 114 fonctions atteintes (cumul 166)
  <= 4 porte(s) : +  88 fonctions atteintes (cumul 254)

Comparaison avec le decompte theorique des configurations :
  s = 1 : configurations majorantes =         48 (fonctions totales = 256)
  s = 2 : configurations majorantes =       5625 (fonctions totales = 256)
  s = 3 : configurations majorantes =    1259712 (fonctions totales = 256)
  s = 4 : configurations majorantes =  466948881 (fonctions totales = 256)

borne de comptage (majorant grossier) : une fonction exige au moins 2 portes
cloture exhaustive mesuree : 254 fonctions sur 256 sont atteignables en <= 4 portes


### Lecture du résultat — le décompte est un majorant, la clôture est un fait

Le décompte des configurations (`3^s (n+s)^{2s}`) est un **majorant** : il compte
des configurations de circuits, dont plusieurs décrivent la même fonction. La
clôture exhaustive, elle, compte des **fonctions** : elle est au plus égale, et
l'écart entre les deux est exactement la redondance (deux circuits, une fonction).

La borne « au moins `s` portes » sort du minorant `3^s (n+s)^{2s} ≥ 2^{2^n}` :
à `n = 3`, la clôture exhaustive montre que la quasi-totalité des 256 fonctions
est déjà atteinte en quatre portes — la borne de comptage est ici **très faible**,
et c'est normal : elle ne devient interessante qu'asymptotiquement. Le carnet la
mesure quand même, pour fixer l'échelle exacte avant l'objet explicite.

In [8]:
# --- L'objet explicite : plan de Fano, graphe de Heawood ---------------------
def plan_de_fano():
    """Points et droites du plan de Fano (F_2^3 prive de {0}).

    Points : les 7 vecteurs non nuls. Droites : les triplets {a, b, a xor b}.
    """
    points = list(range(1, 8))
    droites = sorted(
        {tuple(sorted((a, b, a ^ b))) for a in points for b in points if a < b}
    )
    return points, droites


def graphe_incidence(points, droites):
    """Graphe biparti incidence : sommets points + droites, arites = incidence."""
    sommets = [("p", p) for p in points] + [("d", d) for d in droites]
    aretes = []
    for i, d in enumerate(droites):
        for p in d:
            aretes.append((len(points) + i, points.index(p)))
    return sommets, aretes


def a_un_4_cycle(aretes, n_sommets) -> bool:
    """Vrai s'il existe un cycle de longueur 4 (deux sommets a >= 2 voisins communs)."""
    voisins = {v: set() for v in range(n_sommets)}
    for a, b in aretes:
        voisins[a].add(b)
        voisins[b].add(a)
    for v in range(n_sommets):
        for w in range(v + 1, n_sommets):
            communs = voisins[v] & voisins[w]
            if len(communs) >= 2:
                return True
    return False


def couplage_max(aretes, n_sommets) -> int:
    """Taille d'un couplage maximum (recherche exhaustive par backtracking)."""
    meilleur = [0]

    def explorer(i, utilise, taille):
        if taille + (len(aretes) - i) <= meilleur[0]:
            return
        if i == len(aretes):
            meilleur[0] = max(meilleur[0], taille)
            return
        a, b = aretes[i]
        explorer(i + 1, utilise, taille)
        if a not in utilise and b not in utilise:
            utilise.add(a)
            utilise.add(b)
            explorer(i + 1, utilise, taille + 1)
            utilise.discard(a)
            utilise.discard(b)

    explorer(0, set(), 0)
    return meilleur[0]


points, droites = plan_de_fano()
sommets, aretes = graphe_incidence(points, droites)
E = len(aretes)
tau = couplage_max(aretes, len(sommets))

degres = Counter()
for a, b in aretes:
    degres[a] += 1
    degres[b] += 1

print("Plan de Fano et graphe d'incidence de Heawood :")
print(f"  points            : {len(points)}")
print(f"  droites           : {len(droites)}")
print(f"  sommets           : {len(sommets)}")
print(f"  aretes |E|        : {E}")
print(f"  regularite        : {sorted(set(degres.values()))}")
print(f"  4-cycle present   : {a_un_4_cycle(aretes, len(sommets))}")
print(f"  couplage max tau : {tau}")
print(f"  |E| + tau - 1     : {E} + {tau} - 1 = {E + tau - 1}")

# La matrice d'incidence 7x7 sur GF(2) : son rang est un fait mesure.
M = np.zeros((7, 7), dtype=np.uint8)
for i, d in enumerate(droites):
    for p in d:
        M[i, points.index(p)] = 1


def rang_gf2(m: np.ndarray) -> int:
    m = m.copy() % 2
    r = 0
    for c in range(m.shape[1]):
        piv = None
        for i in range(r, m.shape[0]):
            if m[i, c] == 1:
                piv = i
                break
        if piv is None:
            continue
        m[[r, piv]] = m[[piv, r]]
        for i in range(m.shape[0]):
            if i != r and m[i, c] == 1:
                m[i] ^= m[r]
        r += 1
    return r


print(f"  rang de la matrice d'incidence sur GF(2) : {rang_gf2(M)}")

Plan de Fano et graphe d'incidence de Heawood :


  points            : 7
  droites           : 7
  sommets           : 14
  aretes |E|        : 21
  regularite        : [3]
  4-cycle present   : False
  couplage max tau : 7
  |E| + tau - 1     : 21 + 7 - 1 = 27
  rang de la matrice d'incidence sur GF(2) : 4


### Lecture du résultat — l'objet entier, recalculé dans la cellule

Le plan de Fano est reconstruit **depuis zéro** (les vecteurs non nuls de
`GF(2)^3`, les droites `{a, b, a ⊕ b}`) et tout est **recalculé** : 14 sommets,
21 arêtes, graphe 3-régulier, **absence de 4-cycle** (test : deux sommets à deux
voisins communs), couplage maximum `τ = 7` par backtracking exhaustif. La
formule du registre donne alors `|E| + τ − 1 = 21 + 7 − 1 = 27`.

Le rang de la matrice d'incidence sur GF(2) (4) est un fait supplémentaire du
même objet — il relie la combinatoire du plan à l'algèbre linéaire sur GF(2), donc
aux deux sections précédentes.

**Ce qui n'est pas fait ici** : la conjecture du registre n'est **pas** vérifiée.
Ce que le carnet calcule, c'est la quantité combinatoire `|E| + τ − 1` sur un
objet qu'il construit ; le registre, lui, énonce une borne sur une famille de
circuits **en forme étoile**, restriction que l'on ne réinstalle pas ici. Les deux
n'ont pas le même statut : l'un est **calculé**, l'autre est **cité**.

In [9]:
# --- Temoin 4 (negatif) : un graphe AVEC 4-cycle ---------------------------
def graphe_aleatoire(n_sommets: int, aretes: list[tuple[int, int]]) -> tuple[list, list]:
    return list(range(n_sommets)), aretes


# K_{2,2} : 4 sommets, 4 aretes, un 4-cycle.
sommets_k, aretes_k = graphe_aleatoire(4, [(0, 2), (0, 3), (1, 2), (1, 3)])
tau_k = couplage_max(aretes_k, len(sommets_k))
print("Temoin 4 (negatif) -- K_{2,2} :")
print(f"  |E| = {len(aretes_k)}, tau = {tau_k}, |E| + tau - 1 = {len(aretes_k) + tau_k - 1}")
print(f"  4-cycle present : {a_un_4_cycle(aretes_k, len(sommets_k))}")
print("  => l'hypothese « pas de 4-cycle » de la forme etoile EST VIOLEE :")
print("     la formule ne s'applique pas, et le calcul ne pretend rien dire d'elle.")

# --- Temoin 5 : les deux petits cas releves dans l'archive -----------------
print()
print("Temoin 5 -- les deux petits cas, recalcules :")


def chemin(n_sommets: int):
    aretes = [(i, i + 1) for i in range(n_sommets - 1)]
    return list(range(n_sommets)), aretes


def cycle(n_sommets: int):
    aretes = [(i, (i + 1) % n_sommets) for i in range(n_sommets)]
    return list(range(n_sommets)), aretes


for nom, (sv, ar) in (("P4 (chemin sur 4 sommets)", chemin(4)), ("C6 (cycle sur 6 sommets)", cycle(6))):
    t = couplage_max(ar, len(sv))
    print(f"  {nom:<30} |E| = {len(ar)}, tau = {t}, |E| + tau - 1 = {len(ar) + t - 1}")

Temoin 4 (negatif) -- K_{2,2} :
  |E| = 4, tau = 2, |E| + tau - 1 = 5
  4-cycle present : True
  => l'hypothese « pas de 4-cycle » de la forme etoile EST VIOLEE :
     la formule ne s'applique pas, et le calcul ne pretend rien dire d'elle.

Temoin 5 -- les deux petits cas, recalcules :
  P4 (chemin sur 4 sommets)      |E| = 3, tau = 2, |E| + tau - 1 = 4
  C6 (cycle sur 6 sommets)       |E| = 6, tau = 3, |E| + tau - 1 = 8


### Lecture du résultat — un témoin négatif, deux petits cas provisoires

Le témoin négatif est **décisif** pour la discipline du carnet : sur `K_{2,2}`, la
formule donnerait `4 + 2 − 1 = 5`, mais l'hypothèse « pas de 4-cycle » est
violée. Une formule qui s'applique partout n'est pas une borne ; une formule dont
**l'hypothèse** peut échouer doit être citée avec elle.

Les deux petits cas (`P4` et `C6`) sont **recalculés** et **marqués provisoires** :
ils proviennent du relevé archivé de l'issue, dont l'énoncé exact a été **retiré
avec remplacement** par le registre lui-même (« same term »). Le carnet les
recalcule donc comme un **contrôle de la formule**, pas comme un contrôle de
l'énoncé du registre.

**Rappel de protocole** : le registre réécrit ses énoncés. Toute borne citée
depuis ce flux doit être relue dans sa forme exacte, à la date de son relevé.

## 5. La frontière vivante : lire le registre sans le reprendre

Le protocole du registre, lu dans l'archive :

- une **conjecture** est proposée par un siège, avec un énoncé textuel ;
- elle est **vérifiée** par un juge extérieur (« the checker accepted … on an
  outside runner ») ;
- elle est **réfutée** par la vérification de sa **négation** (« its negation,
  L-meta-(N−1), is checked ») ;
- elle est parfois **retirée avec remplacement et motif** (`lemma_retired: … replaced
  by …: … plain now says … ; same term`) ;
- le flux porte **le texte** de l'énoncé, et le registre **le réécrit**.

La cellule ci-dessous relit l'archive (flux Atom + relevé de l'issue), compte les
catégories, et **vérifie** que la borne calculée dans ce carnet n'est pas présentée
comme la leur : les deux objets sont printés côte à côte, chacun avec sa source.

In [10]:
# --- Lecture de l'archive datee ----------------------------------------------
ARCHIVE = Path(
    r"G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness"
    r"\2026 - Brilliant - pequalnp.com - P vs NP attacked live by a fleet of AI agents (releve date du 25-11-2026)"
)

if not ARCHIVE.exists():
    print("ARCHIVE ABSENTE sur ce poste : le registre n'est pas relu ici.")
    print("Le carnet reste valide : il ne tire aucune conclusion du registre.")
else:
    import re

    flux = (ARCHIVE / "feed.atom").read_text(encoding="utf-8")
    entrees = re.findall(r"<entry>(.*?)</entry>", flux, flags=re.S)
    categories = Counter(re.findall(r'<category term="([^"]+)"', flux))
    print(f"Archive : {ARCHIVE.name}")
    print(f"Entrees dans la fenetre relevee : {len(entrees)}")
    for cat, nb in sorted(categories.items()):
        print(f"  {cat:<18} {nb:>3}")

    # Les entrees qui parlent du plan de Fano : l'objet de la section 4.
    fano = [t for t in re.findall(r"<title>(.*?)</title>", flux, flags=re.S) if "Fano" in t]
    print()
    print("Entrees mentionnant le plan de Fano (verbatim) :")
    for t in fano:
        print("  -", t[:150])

    # Les formes de refute et de retire, relues telles quelles.
    refute = [t for t in re.findall(r"<title>(.*?)</title>", flux, flags=re.S) if t.startswith("Refuted")]
    if refute:
        print()
        print("Forme d'une refutation (verbatim) :", refute[0])

    print()
    print("NOTRE calcul (section 4)  : |E| + tau - 1 = 27 sur le graphe de Heawood reconstruit ici")
    print("LEUR enonce (registre)    : une borne en forme etoile sur une famille de circuits")
    print("=> deux enonces distincts ; le registre n'est ni verifie ni repris par ce carnet.")

Archive : 2026 - Brilliant - pequalnp.com - P vs NP attacked live by a fleet of AI agents (releve date du 25-11-2026)
Entrees dans la fenetre relevee : 30
  lemma_checked       13
  lemma_proposed      13
  lemma_refuted        4

Entrees mentionnant le plan de Fano (verbatim) :
  - Conjecture: seat-intake-1 proposed L-meta-11463: A step toward: Fano plane: a circuit of at most 27 AND/OR gates of fan-in two (distinct earlier wires
  - Conjecture: seat-intake-1 proposed L-meta-11461: A step toward: Fano plane: a circuit of at most 27 AND/OR gates of fan-in two (distinct earlier wires

Forme d'une refutation (verbatim) : Refuted: L-meta-10041 is false: its negation, L-meta-10042, is checked.

NOTRE calcul (section 4)  : |E| + tau - 1 = 27 sur le graphe de Heawood reconstruit ici
LEUR enonce (registre)    : une borne en forme etoile sur une famille de circuits
=> deux enonces distincts ; le registre n'est ni verifie ni repris par ce carnet.


### Lecture du résultat — deux énoncés distincts, aucune confusion

La cellule affiche côte à côte **notre** calcul (`|E| + τ − 1 = 27`, sur un graphe
construit dans le carnet) et **leur** énoncé (une borne sur des circuits en forme
étoile, dans un registre dont le protocole fait autorité sur lui-même et sur
personne d'autre).

Ce que le registre **ne** nous autorise pas à faire : présenter l'une de ses
conjectures comme un résultat acquis, ou son absence de réfutation comme une
démonstration. Ce qu'il **nous** donne : un cas d'étude d'épistémologie en direct,
où la forme de l'énoncé change, où une réfutation est une **vérification de la
négation**, et où un retrait est motivé.

## 6. Exercices

Trois exercices, sur le modèle de la série — **stubs non bloquants** (C.1) : le
carnet s'exécute de bout en bout même les exercices non complétés.

In [11]:
def degre_dune_propriete(f: np.ndarray, p: int = 2) -> int:
    """Exercice 1. Degree de l'extension multilineaire sur GF(p) d'une table.

    Rappel : l'extension multilineaire de f sur GF(p) a pour coefficients
    c_S = Σ_{T ⊆ S} (-1)^{|S|-|T|} f(1_T) (mod p) ; le degré est le plus grand
    |S| tel que c_S ≠ 0.

    TODO etudiant : implementer l'inversion de Moebius sur Z/p et retourner le
    degré. Les fonctions utilitaires du carnet sont dans la section 3.
    """
    result = None  # TODO etudiant
    return result


print("Exercice 1 : degre_dune_propriete renvoie", degre_dune_propriete(np.array([0, 1, 1, 0], dtype=np.uint8)))

Exercice 1 : degre_dune_propriete renvoie None


**Exercice 2** — la borne en forme étoile est une **formule sous hypothèse**.
Reprendre le graphe de Heawood de la section 4, et vérifier par vous-même que
la formule `|E| + τ − 1` **dépend** de l'hypothèse « pas de 4-cycle » :
construire un graphe où elle échoue, et écrire la phrase qui rendrait la citation
honnête.

In [12]:
def borne_etoile(aretes: list[tuple[int, int]], n_sommets: int) -> dict:
    """Exercice 2. Calcule |E| + tau - 1 et teste l'hypothese « pas de 4-cycle ».

    TODO etudiant : renvoyer un dictionnaire avec les cles "E", "tau", "borne" et
    "hypothese_tient" (False si un 4-cycle est present). Le carnet fournit
    couplage_max et a_un_4_cycle.
    """
    result = None  # TODO etudiant
    return result


print("Exercice 2 : borne_etoile renvoie", borne_etoile([(0, 2), (0, 3), (1, 2), (1, 3)], 4))

Exercice 2 : borne_etoile renvoie None


**Exercice 3** — le mur du théorème de Razborov–Rudich est une colonne, pas une
mesure. Reprendre le tableau des densités de la section 2 et ajouter une
**quatrième colonne** : « la propriété est-elle *utile* ? ». Dire précisément ce
qu'un tiers devrait fournir pour y répondre, et pourquoi la mesure du carnet ne
peut pas y répondre.

In [13]:
def colonnes_utilite(mesurees: dict) -> list[str]:
    """Exercice 3. Colonne « utile ? » du tableau des propriétés.

    TODO etudiant : renvoyer, pour chaque propriété, une phrase qui dit soit
    « non : la propriété est une tautologie ou sa négation l'est », soit « le
    mur : troisième colonne non atteinte, le théorème est cité et non re-prouvé ».
    """
    result = None  # TODO etudiant
    return result


print("Exercice 3 : colonnes_utilite renvoie", colonnes_utilite(mesurees))

Exercice 3 : colonnes_utilite renvoie None


## 7. Conclusion

| Geste | Où | Ce qui a été fait | Niveau |
|---|---|---|---|
| Trois barrières attribuées | §1 | Relativisation (rappelée, traitée en 03/03c), natural proofs, algébrisation | **cité** |
| Densité et coût de calcul | §2 | Population **complète** des 65 536 fonctions à 4 variables : parité, degrés, sensibilité ; famille pseudo-aléatoire (K = 256) ; loi de référence (200 tirages aléatoires) | **mesuré** |
| Degré d'extension multilinéaire | §3 | AND, OR, PARITY, MAJORITY : degrés sur GF(2), GF(3), GF(5) et nombre de portes d'un circuit **construit** | **mesuré** |
| Jouet d'oracles | §3 | Deux mondes de même densité et de même degré, séparés par un test algébrique | **mesuré** |
| Comptage et clôture exhaustive | §4 | `n = 3`, portee des configurations contre fonctions atteintes en ≤ 4 portes | **mesuré** |
| Objet explicite | §4 | Plan de Fano, graphe de Heawood, `arêtes = 21`, `τ = 7`, `21 + 7 − 1 = 27`, rang 4 sur GF(2) | **mesuré** |
| Témoin négatif | §4 | `K_{2,2}` : l'hypothèse « pas de 4-cycle » est violée | **mesuré** |
| Deux petits cas | §4 | `P4` → 4, `C6` → 8 — **provisoires**, énoncé du registre relu dans l'archive | **recalculé** |
| Registre du site archivé | §5 | Fenêtre relue, formes de réfutation citées, énoncés non repris | **lu** |

## À retenir

1. **Trois barrières, trois régimes.** La relativisation demande que la preuve
   tienne dans deux mondes ; l'algébrisation en ajoute un, polynomial ; les natural
   proofs disent qu'un certain type de preuve **ne peut pas** exister sous hypothèse
   forte. La série traitait le premier ; ce carnet nomme les deux autres.
2. **Une borne se cite avec son hypothèse.** Sur `K_{2,2}`, la formule donne un
   nombre et l'hypothèse est violée : c'est le témoin négatif, pas un accident.
3. **Un registre qui réécrit ses énoncés** impose de dater ses citations. Le
   carnet lit l'archive datée, et ne présente rien de ce qu'elle contient comme un
   résultat acquis.
4. **Ce que ce carnet ne prouve pas** : ni `P ≠ NP`, ni la borne du registre, ni les
   théorèmes cités (Razborov–Rudich, Aaronson–Wigderson). Les hypothèses de ces
   théorèmes sont **attribuées**, non vérifiées ici.

## Vérifiez votre compréhension

1. Pourquoi la parité est-elle **grande** et **constructive**, mais pas **utile** ?
   Que faudrait-il construire pour rendre une propriété utile ?
2. Dans la section 3, pourquoi le degré d'une fonction booléenne est-il **indépendant
   du corps** (GF(2), GF(3), GF(5)) ? Que se passerait-il si l'on prenait une
   fonction à valeurs dans `F_p` qui n'est pas booléenne ?
3. Le témoin négatif `K_{2,2}` : quelle phrase faut-il écrire dans une PR qui cite
   la borne du registre, pour que la citation soit honnête ?
4. Le flux dit « its negation, L-meta-(N−1), is checked ». En quoi une réfutation
   obtenue par la négation est-elle plus solide qu'un échec du vérificateur ?

## Pour aller plus loin

- **Carnets voisins** : [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb),
  [03c — Complexité Zoo](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb),
  [03d — L'ETH et sa réfutation annoncée](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb),
  [03e — Barrières d'exposants](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb),
  [05 — Compter est plus dur que vérifier](Complexity-05-CountingHarder-Permanent-Python.ipynb),
  [05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb).
- **Le même objet, ailleurs dans le dépôt** : le graphe de Heawood est déjà l'objet
  de l'exercice 3 de
  [Frontieres-01 — Edge coloring et Tutte](../../Search/Part5-Frontieres/Frontieres-01-EdgeColoring-Tutte-Python.ipynb) ;
  le couplage hongrois est implémenté par
  [Sudoku-15](../../Sudoku/Sudoku-15-Infer-CSharp.ipynb) et la permanente
  est l'objet de `05`/`05b` — **le même objet combinatoire, trois régimes : compter,
  optimiser, borner**.
- **Sources**, toutes archivées dans le gisement partagé
  (`Bibliothèque IA/Automata/Complexity Theory and Hardness/`) :
  - Baker–Gill–Solovay, *Relativizations of the P versus NP Question*,
    SIAM J. Comput. 4(4), 1975 ;
  - Razborov–Rudich, *Natural Proofs*, STOC '94 ; version journal
    **J. Comput. Syst. Sci. 55(1):24–35, 1997** (copie archivée : manuscrit de
    l'auteur daté du 30-11-1996) ;
  - Aaronson–Wigderson, *Algebrization: A New Barrier in Complexity Theory*,
    STOC 2008 ; version étendue **ACM Transactions on Computation Theory 1(1),
    2009** — dite *TOCT* dans le tableau de la section 1 — (copie archivée :
    résumé étendu STOC, 10 pages).
- **Pourquoi ces références ont été relues sur les copies** : une identification
  mémorisée se trompe — `arXiv:0801.0976` ne désigne **pas** « Algebrization » (il
  renvoie à un autre article), et la version de revue de *Natural Proofs* est de
  1997, non de 1994. Les titres, auteurs et éditions ont donc été lus sur la
  première page des copies archivées.